# Phase 6 — Investigation Agent Validation

The orchestrator that turns single-shot RAG into an **investigation**:

```
plan  →  retrieve (cross-stream + dependency graph)  →  grade evidence
                        ↑______ re-search if thin ______|
                                                        → synthesize (cited answer)
```

Built on LangGraph; the LLM (Gemini) drives plan + grade, retrieval + graph are deterministic tools. **Needs GEMINI_API_KEY.**

In [ ]:
import os, sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "src"))

from archaeologist.agent.graph import investigate
from archaeologist.rag.llm import has_api_key, active_model
print("LLM:", active_model(), "| key set:", has_api_key())

In [ ]:
def run(question, max_iterations=2):
    r = investigate(question, max_iterations=max_iterations)
    print(f"Q: {question}\n")
    print("--- investigation trace ---")
    for step in r["trace"]:
        print("  •", step)
    print("\n--- evidence used ---")
    for i, e in enumerate(r["evidence"], 1):
        print(f"  [{i}] ({e['stream']:6}) {e['citation']:26.26} {e['title'][:42]}")
    print("\n--- answer ---\n")
    print(r["answer"])
    return r

## An impact question — should trigger the dependency graph
Naming a symbol lets the planner set a `graph_target`, so a `graph` evidence item (what-breaks / what-it-calls) joins the retrieved code and docs.

In [ ]:
_ = run("What would break if I removed Flask.dispatch_request, and what does it call?")

## A 'why' question — should pull commits + docs, not just code

In [ ]:
_ = run("Why did Flask move away from LocalStack for its context handling?")

## Summary
The agent plans an evidence strategy, gathers across all streams **plus** the dependency graph, judges sufficiency, re-searches gaps (up to `max_iterations`), and answers with citations — the full 'investigation path' from the project vision.